# 02 - Cleaning, RIASEC scores and major profiling
Reads `RIASEC_data12Dec2018/data.csv` (never modified) and writes only to `processed/`. No grouping of majors, no train/test split, no model.

## 0. Setup
We read every column as text (`dtype=str`) so nothing is silently converted to NaN, then convert the columns we need. Seed 42 is set for reproducibility (nothing below is random).

In [1]:
from pathlib import Path
import re, random, hashlib
import numpy as np, pandas as pd
random.seed(42); np.random.seed(42)
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 400)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "RIASEC_data12Dec2018" / "data.csv"
OUT = ROOT / "processed"; OUT.mkdir(exist_ok=True)
sha_before = hashlib.sha256(SRC.read_bytes()).hexdigest()

header = open(SRC, encoding="utf-8", errors="replace").readline()
delim = max([",", "\t", ";", "|"], key=header.count)
raw = pd.read_csv(SRC, sep=delim, dtype=str, keep_default_na=False, encoding_errors="replace")
raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]       # one malformed row created a stray column
raw.insert(0, "row_id", np.arange(len(raw)))                    # position in the original file, for traceability
print("delimiter:", repr(delim), "| shape (with row_id):", raw.shape)

delimiter: '\t' | shape (with row_id): (145828, 94)


## 1. Check the codebook first

In [2]:
book = open(SRC.parent / "codebook.txt", encoding="utf-8", errors="replace").read().splitlines()
for line in book:
    if re.match(r"(uniqueNetworkLocation|education)\s", line) or re.match(r"VCL(6|9|12)\s", line) or "not real words" in line:
        print(line.strip())
print()
print("uniqueNetworkLocation values in data:", raw["uniqueNetworkLocation"].value_counts().to_dict())
print("education values in data           :", raw["education"].value_counts().sort_index().to_dict())

VCL6	cuivocal
VCL9	florted
VCL12	verdid
A value of 1 is checked, 0 means unchecked. The words at VCL6, VCL9, and VCL12 are not real words and can be used as a validity check.
education			"How much education have you completed?", 1=Less than high school, 2=High school, 3=University degree, 4=Graduate degree
uniqueNetworkLocation	1 if the record is the only one from its network location in the dataset, 2 if there are more than one record. There can be more than one record from the same network if for example that network is shared by a school etc, or it may be because of test retakes

uniqueNetworkLocation values in data: {'1': 100727, '2': 45101}
education values in data           : {'0': 1125, '1': 22665, '2': 62625, '3': 39102, '4': 20311}


**Findings.**
- `uniqueNetworkLocation` is a **yes/no flag**, not an ID: `1` = the only record from that network, `2` = more than one record. So we cannot group by network; we add a `test_eligible` column instead (step 4).
- Fake words are `VCL6` (cuivocal), `VCL9` (florted), `VCL12` (verdid).
- `education`: 1 = less than high school, 2 = high school, **3 = university degree, 4 = graduate degree**. The value 0 appears in the data but is not documented (treated as no answer).

## 2. Filters, applied in order, with the row count logged after each
Helper `log()` records each step so we can print the funnel table at the end.

In [3]:
ITEMS = [f"{l}{i}" for l in "RIASEC" for i in range(1, 9)]
FAKE = ["VCL6", "VCL9", "VCL12"]
funnel = [("0. original file", len(raw))]
def log(name, df):
    funnel.append((name, len(df))); print(f"{name:<55s} {len(df):>8,}")
    return df

df = raw.copy()
for c in ITEMS + FAKE + ["age", "education"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# a. adults, and remove impossible ages
df = log("a. age 18-100", df[(df["age"] >= 18) & (df["age"] <= 100)])
# b. university degree (3) or graduate degree (4)
df = log("b. education = university or graduate degree", df[df["education"].isin([3, 4])])
# c. fake-word check: drop anyone who ticked ANY fake word (1 = ticked)
df = log("c. passed fake-word check (VCL6/9/12 all unticked)", df[(df[FAKE] == 0).all(axis=1)])
# d. RIASEC items: 0 means "no answer" -> missing; drop any row with a missing item
df[ITEMS] = df[ITEMS].replace(0, np.nan)
df = log("d. all 48 RIASEC items answered (0 = missing)", df.dropna(subset=ITEMS))

a. age 18-100                                            108,217
b. education = university or graduate degree              59,125
c. passed fake-word check (VCL6/9/12 all unticked)        46,490
d. all 48 RIASEC items answered (0 = missing)             43,602


### Normalising the major text (needed to spot placeholders and used again in step 5)
Rule: lowercase, drop apostrophes, turn every other punctuation mark into a space, collapse repeated spaces, trim. So `"N/A"` becomes `"n a"` and `"Psychology "` becomes `"psychology"`. Nothing is merged beyond that: `"comp sci"` and `"computer science"` stay different strings.

In [4]:
def norm_major(s):
    s = s.lower().replace("'", "").replace("’", "")
    s = re.sub(r"[^\w\s]", " ", s).replace("_", " ")
    return re.sub(r"\s+", " ", s).strip()

df["major_raw"] = df["major"] if "major" in df else raw.loc[df.index, "major"]
df["major_norm"] = df["major_raw"].map(norm_major)

# Placeholder list: values that are not a field of study. Chosen by looking at what actually occurs in
# the data (see counts below). "aa" is NOT included (may be the real degree Associate of Arts);
# "general" and "other" are ambiguous and are kept, see the report.
PLACEHOLDERS = {"no", "none", "na", "n a", "nothing", "nil", "null", "nope", "nan", "yes", "n", "x", "xx", "xxx",
                "undecided", "undeclared", "unknown", "idk", "dont know", "not sure", "unsure", "test"}
is_placeholder = (
    (df["major_norm"] == "")                      # empty, or only punctuation (e.g. "-", "...")
    | df["major_norm"].isin(PLACEHOLDERS)
    | (df["major_norm"].str.len() == 1)           # single characters
    | df["major_norm"].str.fullmatch(r"\d+")      # digits only
)
print("Placeholder strings (explicit list):", sorted(PLACEHOLDERS))
print("Plus rules: empty/punctuation-only, single character, digits only\n")
why = df.loc[is_placeholder, "major_norm"].replace("", "(empty / punctuation only)")
print("Rows removed by this step, by value:")
print(why.value_counts().to_string())
df = log("e. major present and not a placeholder", df[~is_placeholder])

Placeholder strings (explicit list): ['dont know', 'idk', 'n', 'n a', 'na', 'nan', 'nil', 'no', 'none', 'nope', 'not sure', 'nothing', 'null', 'test', 'undecided', 'undeclared', 'unknown', 'unsure', 'x', 'xx', 'xxx', 'yes']
Plus rules: empty/punctuation-only, single character, digits only

Rows removed by this step, by value:
major_norm
(empty / punctuation only)    1540
no                              55
undecided                       16
yes                             10
none                             7
n a                              7
na                               4
undeclared                       2
l                                2
xx                               1
unknown                          1
unsure                           1
p                                1
55                               1
not sure                         1
e                                1
idk                              1
dont know                        1
e. major present and not a plac

In [5]:
# f. exact duplicates on all 48 items + age + major (keep the first copy)
key = ITEMS + ["age", "major_norm"]
dup_norm = df.duplicated(subset=key, keep="first")
dup_raw  = df.duplicated(subset=ITEMS + ["age", "major_raw"], keep="first")
print(f"duplicates found using normalised major: {dup_norm.sum():,}  (using raw major it would be {dup_raw.sum():,})")
df = log("f. exact duplicates removed (items + age + major)", df[~dup_norm])

duplicates found using normalised major: 3  (using raw major it would be 3)
f. exact duplicates removed (items + age + major)         41,947


In [6]:
f = pd.DataFrame(funnel, columns=["step", "rows remaining"])
f["rows removed"] = (-f["rows remaining"].diff()).fillna(0).astype(int)
f["% of original"] = (f["rows remaining"] / funnel[0][1] * 100).round(1)
print(f.to_string(index=False))

                                              step  rows remaining  rows removed  % of original
                                  0. original file          145828             0          100.0
                                     a. age 18-100          108217         37611           74.2
      b. education = university or graduate degree           59125         49092           40.5
c. passed fake-word check (VCL6/9/12 all unticked)           46490         12635           31.9
     d. all 48 RIASEC items answered (0 = missing)           43602          2888           29.9
            e. major present and not a placeholder           41950          1652           28.8
 f. exact duplicates removed (items + age + major)           41947             3           28.8


## 3. RIASEC scale scores
Each scale score is the **mean of its 8 items** (range 1 to 5). Using the mean (not the sum) keeps the scale readable and independent of how many items a scale has.

In [7]:
for L in "RIASEC":
    df[f"{L}_score"] = df[[f"{L}{i}" for i in range(1, 9)]].mean(axis=1)
SC = [f"{L}_score" for L in "RIASEC"]
print(df[SC].describe().round(3).T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].to_string())
corr = df[SC].corr()
corr.index = corr.columns = list("RIASEC")
print("\nCorrelation matrix (Pearson):")
print(corr.round(3).to_string())

           count   mean    std  min    25%    50%    75%  max
R_score  41947.0  2.104  0.860  1.0  1.375  2.000  2.750  5.0
I_score  41947.0  3.039  1.010  1.0  2.250  3.125  3.875  5.0
A_score  41947.0  2.984  0.964  1.0  2.250  3.000  3.750  5.0
S_score  41947.0  3.402  0.851  1.0  2.875  3.500  4.000  5.0
E_score  41947.0  2.536  0.856  1.0  1.875  2.500  3.125  5.0
C_score  41947.0  2.417  0.964  1.0  1.625  2.375  3.125  5.0

Correlation matrix (Pearson):
       R      I      A      S      E      C
R  1.000  0.332  0.174  0.037  0.296  0.457
I  0.332  1.000  0.330  0.126  0.009  0.053
A  0.174  0.330  1.000  0.281  0.251 -0.068
S  0.037  0.126  0.281  1.000  0.351  0.111
E  0.296  0.009  0.251  0.351  1.000  0.447
C  0.457  0.053 -0.068  0.111  0.447  1.000


In [8]:
# Holland's hexagon order is R-I-A-S-E-C around a circle. Distance 1 = neighbours (RI, IA, AS, SE, EC, RC),
# distance 2 = one type apart (RA, IS, AE, SC, RE, IC), distance 3 = opposites (RS, IE, AC).
# (pairs are labelled in R-I-A-S-E-C order, so "RC" means R with C and "RE" means R with E)
order = list("RIASEC")
rows = []
for i in range(6):
    for j in range(i + 1, 6):
        d = min(j - i, 6 - (j - i))
        rows.append((order[i] + order[j], d, corr.loc[order[i], order[j]]))
pairs = pd.DataFrame(rows, columns=["pair", "hexagon distance", "r"])
print(pairs.sort_values(["hexagon distance", "pair"]).round(3).to_string(index=False))
print("\nMean r by hexagon distance:")
print(pairs.groupby("hexagon distance")["r"].agg(["count", "mean"]).round(3).to_string())

pair  hexagon distance      r
  AS                 1  0.281
  EC                 1  0.447
  IA                 1  0.330
  RC                 1  0.457
  RI                 1  0.332
  SE                 1  0.351
  AE                 2  0.251
  IC                 2  0.053
  IS                 2  0.126
  RA                 2  0.174
  RE                 2  0.296
  SC                 2  0.111
  AC                 3 -0.068
  IE                 3  0.009
  RS                 3  0.037

Mean r by hexagon distance:
                  count   mean
hexagon distance              
1                     6  0.366
2                     6  0.169
3                     3 -0.007


## 4. Test-set eligibility (no split yet)
`uniqueNetworkLocation` is a flag, so `test_eligible = True` only where it equals 1 (the record was the only one from its network in the *original* 145,828-row file, which is the stricter check).

In [9]:
df["test_eligible"] = df["uniqueNetworkLocation"] == "1"
n_el = int(df["test_eligible"].sum())
print(f"test-eligible rows: {n_el:,} of {len(df):,} ({n_el/len(df):.1%}); not eligible: {len(df)-n_el:,}")

test-eligible rows: 34,657 of 41,947 (82.6%); not eligible: 7,290


## 5. Major profiling (observation only, nothing is merged)

In [10]:
vc = df["major_norm"].value_counts()
print("clean rows:", len(df), "| distinct normalised majors:", len(vc), "| appearing once:", int((vc == 1).sum()))
top = pd.DataFrame({"rank": range(1, len(vc) + 1), "major_norm": vc.index, "count": vc.values})
top["pct"] = (top["count"] / len(df) * 100).round(3)
top["cum_pct"] = (top["count"].cumsum() / len(df) * 100).round(2)
print(top.head(300).to_string(index=False))

clean rows: 41947 | distinct normalised majors: 5551 | appearing once: 4102
 rank                                major_norm  count    pct  cum_pct
    1                                psychology   6422 15.310    15.31
    2                                  business   1656  3.948    19.26
    3                                   english   1470  3.504    22.76
    4                                 education   1104  2.632    25.39
    5                                   biology    852  2.031    27.43
    6                                 economics    723  1.724    29.15
    7                                accounting    602  1.435    30.58
    8                          computer science    602  1.435    32.02
    9                               engineering    588  1.402    33.42
   10                                   nursing    585  1.395    34.82
   11                                       law    574  1.368    36.18
   12                                 sociology    565  1.347    37.53
 

In [11]:
cov = pd.DataFrame({"top N majors": [10, 25, 50, 100, 200, 300, 500, 1000, 2000]})
cov["rows covered"] = [int(top["count"].head(n).sum()) for n in cov["top N majors"]]
cov["% of rows"] = (cov["rows covered"] / len(df) * 100).round(1)
print(cov.to_string(index=False))

 top N majors  rows covered  % of rows
           10         14604       34.8
           25         21089       50.3
           50         26186       62.4
          100         29743       70.9
          200         32568       77.6
          300         33886       80.8
          500         35329       84.2
         1000         36947       88.1
         2000         38396       91.5


In [12]:
# Variant families: OBSERVATIONS only (loose patterns, to see how scattered the spellings are).
fam = {
 "computer science":      r"comp.*sci|^cs$|^compsci",
 "information tech/systems": r"^(?:it|i t|ict|cis|mis|information (?:technology|systems).*)$",
 "psychology":            r"^psy",
 "business":              r"^(?:business|bba|mba)",
 "nursing":               r"nurs",
 "education/teaching":    r"^(?:education|elementary education|secondary education|teaching)",
 "english":               r"^english",
 "biology":               r"^bio",
 "engineering":           r"engineer",
 "law":                   r"^(?:law|pre law|prelaw|legal studies)",
 "accounting":            r"^(?:accounting|accountancy|cpa)",
}
for name, pat in fam.items():
    m = vc[vc.index.str.contains(pat, regex=True)]
    print(f"\n{name}: {len(m)} distinct strings, {int(m.sum()):,} rows ({m.sum()/len(df):.1%}); top variants:")
    print("   " + "; ".join(f"{k} ({v})" for k, v in m.head(10).items()))


computer science: 54 distinct strings, 703 rows (1.7%); top variants:
   computer science (602); computer science engineering (17); computer sciences (8); cs (8); computer science and engineering (7); comp sci (5); computing science (4); computer science mathematics (2); computer sciense (2); mathematics and computer science (2)

information tech/systems: 21 distinct strings, 340 rows (0.8%); top variants:
   information technology (147); it (123); information systems (25); mis (11); information systems management (8); ict (6); i t (5); cis (2); information systems technology (1); information technology ebusiness (1)

psychology: 248 distinct strings, 6,926 rows (16.5%); top variants:
   psychology (6422); psycology (22); psych (22); psychology and sociology (18); psycholgy (13); psychology counseling (13); psychology sociology (12); psychology and criminology (10); psychology education (9); psychology english (8)

business: 172 distinct strings, 2,828 rows (6.7%); top variants:
   bu

In [13]:
# Other things worth knowing before hand-mapping
multi = df["major_raw"].str.contains(r"[,/&+;]| and ", case=False, regex=True)
print(f"rows that look like multiple majors (contain , / & + ; or 'and'): {multi.sum():,} ({multi.mean():.1%})")
print("examples:", df.loc[multi, "major_raw"].head(8).tolist())
vague = ["general", "other", "science", "sciences", "arts", "art", "engineering", "business", "liberal arts", "general studies", "humanities", "social science", "health", "education"]
print("\nvague or broad strings (kept):", {k: int(vc.get(k, 0)) for k in vague if vc.get(k, 0)})
print("still ambiguous: 'aa' =", int(vc.get("aa", 0)), "| 'it' =", int(vc.get("it", 0)), "| 'ba' =", int(vc.get("ba", 0)))
nonascii = df["major_norm"].str.contains(r"[^\x00-\x7f]", regex=True)
print("strings with non-English characters:", int(nonascii.sum()))

rows that look like multiple majors (contain , / & + ; or 'and'): 2,899 (6.9%)
examples: ['Plant Science, Genetics', 'Russian Language and Literature', 'career and technical education: workforce and', 'mathmatics, physics, chemistry', 'library&info science', 'Business & Criminal Justice', 'Psychology and counseling', 'psycholgy and film']

vague or broad strings (kept): {'general': 22, 'other': 2, 'science': 221, 'sciences': 14, 'arts': 82, 'art': 208, 'engineering': 588, 'business': 1656, 'liberal arts': 120, 'general studies': 97, 'humanities': 50, 'social science': 113, 'health': 57, 'education': 1104}
still ambiguous: 'aa' = 3 | 'it' = 123 | 'ba' = 7
strings with non-English characters: 18


## 6. Save outputs (to `processed/` only)

In [14]:
keep_cols = (["row_id"] + ITEMS + SC +
             ["age", "gender", "education", "country", "major_raw", "major_norm", "uniqueNetworkLocation", "test_eligible"])
df[ITEMS] = df[ITEMS].astype(int)
clean = df[keep_cols].reset_index(drop=True)
clean.to_csv(OUT / "riasec_clean.csv", index=False, encoding="utf-8")
top.to_csv(OUT / "major_counts.csv", index=False, encoding="utf-8")
chk = pd.read_csv(OUT / "riasec_clean.csv", keep_default_na=False)
print("riasec_clean.csv:", chk.shape, "| major_counts.csv:", len(top), "rows")
print("any missing in numeric columns:", int(chk[ITEMS + SC + ["age"]].isna().sum().sum()), "| empty majors:", int((chk["major_norm"] == "").sum()))
print("original data.csv unchanged:", hashlib.sha256(SRC.read_bytes()).hexdigest() == sha_before)

riasec_clean.csv: (41947, 63) | major_counts.csv: 5551 rows
any missing in numeric columns: 0 | empty majors: 0
original data.csv unchanged: True
